# Confidence routing with Laya

A System 1 model answers fast and reports a confidence.
The routing idea says: trust it when it is sure, escalate when it is not.

That only works if the confidence is honest.
This notebook runs the ladder live and then checks whether it is.
If the network fails, every live cell falls back to the last run saved in `data/`.

In [ ]:
import pandas as pd
from IPython.display import display

from laya_demo import report
from laya_demo.cache import load_run, save_run
from laya_demo.calibration import plot_reliability
from laya_demo.routing import make_tiers, tier_bounds

LIVE = True  # set to False to replay the last cached run from data/ with no network

## The ladder

Nandakishor M, *Confidence-Aware Routing for LLM Reliability Enhancement* (arXiv 2510.01237), assesses confidence before generating and routes on four tiers.
This demo has no retrieval step, so both middle tiers go to Gemini as System 2.

In [ ]:
tiers = make_tiers(high=0.75, mid=0.55, low=0.35)
bounds = tier_bounds(tiers)
pd.DataFrame(
    [{"tier": t.name, "confidence": bounds[t.name], "paper says": t.paper_action, "demo does": t.action} for t in tiers]
)

## The task

Banking77: short customer messages, 77 intents.
The talk cites it because Laya's own README reports 0.425 there against Jev's 0.870.

In [ ]:
from laya_demo.data import load_banking77

N, SEED = 150, 0
if LIVE:
    try:
        items, labels = load_banking77(N, SEED)
    except Exception as exc:  # noqa: BLE001 - any failure here means switch to replay
        print(f"Could not load the dataset ({type(exc).__name__}); switching to replay.")
        LIVE = False
if LIVE:
    print(f"{len(items)} items, {len(labels)} labels")
    display(pd.DataFrame([vars(i) for i in items[:5]]))

## System 1 is Laya

One typed `choice` question, one forward pass, no text generated.
Note the two numbers: `confidence` is one minus normalized entropy, not the probability of the answer.

In [ ]:
from laya_demo.system1 import LayaSystem1

laya = None
if LIVE:
    try:
        laya = LayaSystem1(labels, model="english")
        laya.warm_up()
    except Exception as exc:  # noqa: BLE001 - any failure here means switch to replay
        print(f"Could not load Laya ({type(exc).__name__}: {exc}); switching to replay.")
        LIVE = False
if LIVE:
    example = items[0]
    raw = laya.router.predict({"message": example.text}, laya.question, model="english")
    answer = raw["answers"]["intent"]
    print("message    :", example.text)
    print("gold       :", example.label)
    print("Laya says  :", answer["choice"])
    print("confidence :", answer["confidence"])
    print("top prob   :", max(answer["probabilities"].values()))
    print("routing    :", raw["routing"]["reason"])

## Run the ladder

Laya answers every item, each item lands in a tier, and escalated items go to Gemini.
If anything fails here, the next cell replays the last saved run.

In [ ]:
from laya_demo.pipeline import Config, run_pipeline

run = None
if LIVE:
    try:
        run = run_pipeline(Config(n=N, seed=SEED, max_gemini_calls=60, gemini_baseline=30), system1=laya)
        print("saved to", save_run(run))
    except Exception as exc:  # noqa: BLE001 - fall back to the cached run
        print(f"Live run failed ({type(exc).__name__}: {exc}); replaying the last saved run instead.")
if run is None:
    run = load_run()
print(
    f"run from {run.meta['started_at']}, {run.meta['n']} items, {run.meta['duration_s']} s, Laya on {run.meta['device']}"
)
print("System 2:", run.meta["gemini_status"])

## The trade off

Laya alone is fast and weak here.
Gemini alone is slower.
The routed system spends Gemini only where Laya is unsure, and sends the least sure items to a person.

In [ ]:
pd.DataFrame(report.tradeoff_table(run)).set_index("system").style.format(
    {"accuracy": "{:.1%}", "coverage": "{:.1%}", "mean_latency_ms": "{:,.1f} ms"}, na_rep="-"
)

## Confidently wrong

These are the answers Laya was most sure of and got wrong.
Each one was accepted without review.

In [ ]:
wrong = sorted((r for r in run.records if not r.laya_correct), key=lambda r: -r.laya_confidence)[:6]
pd.DataFrame(
    [
        {"message": r.text, "gold": r.gold, "Laya": r.laya_answer, "confidence": r.laya_confidence, "tier": r.tier}
        for r in wrong
    ]
)

## Per tier

If confidence were honest, the accept tier's accuracy would sit at or above 0.75.

In [ ]:
pd.DataFrame(report.tier_table(run.records, run.tiers)).drop(columns=["gemini_calls"]).set_index("tier").style.format(
    {"share": "{:.1%}", "laya_acc": "{:.1%}", "final_acc": "{:.1%}", "mean_latency_ms": "{:,.1f} ms"}, na_rep="-"
)

## Is the confidence honest?

A reliability diagram bins answers by stated confidence and plots how often they were right.
Bars on the diagonal mean honest confidence.
The number under each bar is how many answers fell in that bin.

In [ ]:
import matplotlib.pyplot as plt

correct = [r.laya_correct for r in run.records]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_reliability([r.laya_confidence for r in run.records], correct, ax=axes[0], label="Laya confidence field")
plot_reliability([r.laya_top_prob for r in run.records], correct, ax=axes[1], label="Laya top probability")
fig.tight_layout()
cal = report.calibration_summary(run.records)
print(f"accuracy {cal['accuracy']:.1%}, mean confidence {cal['mean_confidence']:.3f}, ECE {cal['ece_confidence']:.3f}")